# A graph neural network for molecules (message passing)

*chemistry · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/molecular_gnn)

A molecule is a graph, and a message-passing neural network is a few einsums over its adjacency matrix: every atom repeatedly sums its neighbours' states, $s_{mic} = A_{mij}h_{mjc}$, and the molecule's prediction is a sum over atoms. Trained to predict the Hückel π-electron energy of random conjugated hydrocarbons, it gives the same answer however the atoms are numbered, and it still works on molecules larger than any it was trained on, where a plain MLP on the adjacency matrix fails.

**Vector form**

$$y = \sum_k o_k\lambda_k(A),\qquad \mathbf h_i^{(l+1)} = \sigma\Big(U\,\mathbf h_i^{(l)} + V\!\!\sum_{j\in\mathcal N(i)}\!\mathbf h_j^{(l)} + \mathbf c\Big),\qquad \hat y = \sum_i e\big(\mathbf h_i^{(L)}\big),\qquad \hat y(\Pi A\Pi^{\mathsf T}, \Pi X) = \hat y(A, X)$$

**Index (tensor) form**

$$h^0_{mic} = \sigma(x_{mif}W_{fc} + b_c),\qquad s_{mic} = A_{mij}\,h_{mjc},\qquad h'_{mid} = h_{mid} + \sigma\big(h_{mic}U_{cd} + s_{mic}V_{cd} + c_d\big),\qquad \hat y_m = \mathrm{mask}_{mi}\,e_{mi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{mij}$ | `data["A"]` | (M, N, N) | adjacency of the carbon skeleton of molecule $m$, padded with zeros to $N = 16$ atoms |
| $x_{mif}$ | `features(A, mask)` | (M, N, 3) | atom features: one-hot number of carbon neighbours (1, 2 or 3) |
| $y_m = o_k\lambda_{mk}$ | `huckel_energy(A, mask)` | (M,) | the target: occupied Hückel eigenvalues of $A$ (eigvalsh), π energy in units of $/\beta/$ |
| $h^0_{mic}$ | `einsum('mif,fc->mic', x, W)` | (M, N, C) | $f$ summed: embed each atom into $C$ channels |
| $s_{mic} = A_{mij}h_{mjc}$ | `einsum('mij,mjc->mic', A, h)` | (M, N, C) | repeated $j$ ⇒ summed over the neighbours of atom $i$: the message |
| $h_{mic}U_{cd} + s_{mic}V_{cd}$ | `einsum('mic,cd->mid', h, U) + einsum('mic,cd->mid', s, V)` | (M, N, C) | $c$ summed: the same weights for every atom of every molecule |
| $e_{mi}$ | `atom_contributions(h, Q1, q1, Q2, q2)` | (M, N) | a learned energy per atom (the colours on the molecule) |
| $\hat y_m = \mathrm{mask}_{mi}e_{mi}$ | `e.sum(-1)` | (M,) | $i$ summed over real atoms: size-extensive and independent of the atom order |
| $\Pi A \Pi^{\mathsf T}$ | `A[:, perm][:, :, perm]` | (M, N, N) | relabelling the atoms: the GNN output stays the same to float precision |
| $\mathrm{MLP}(\mathrm{vec}\,A_m)$ | `mlp(data, w)` | (M,) | baseline: a fixed-size network on the 256 flattened adjacency entries |

The data are synthetic on purpose, so that the right answer is known exactly. Each "molecule" is a random hydrogen-suppressed carbon skeleton (a tree with at most three neighbours per carbon, closed into some 5- and 6-membered rings): graph-theoretic toys, not a curated set of real compounds. Its label is the Hückel π-electron energy: with $H = \alpha + \beta A$ and one π electron per carbon filling the orbitals of highest eigenvalue $\lambda_k$ of $A$, $E_\pi = N\alpha + \beta\,\sum_k o_k\lambda_k$, and the network learns $y = \sum_k o_k\lambda_k$ from the graph alone. It never sees an eigenvalue. The tensor lesson is that a graph neural network needs only one new operation, the einsum over the adjacency $A_{mij}h_{mjc}$, which sums the states of the neighbours of every atom in every molecule at once. All weights act on one atom at a time, and atoms are combined only through $A$ and the final sum, so the model is permutation invariant by construction and the same weights fit a molecule of any size. The console shows both properties: relabelling the atoms changes the GNN output by about $10^{-6}$ (float32 rounding) and the MLP output by whole units of $|\beta|$; on molecules with 11–16 carbons, larger than any in training, the MLP has never seen anything in the extra rows of its input and fails, while the GNN error barely grows. Compare with the "bond counting" baseline $aN + bN_{bonds} + c$: π energy is nearly additive (about $1.2|\beta|$ per carbon), and the GNN's real job is the corrections from branching, rings and odd electrons, where it roughly halves the error. Four rounds of messages only reach four bonds, so effects that depend on the whole conjugated system (aromaticity, 4n-rings) are learned only approximately. This is the message-passing neural network (MPNN) framework of Gilmer et al. (ICML 2017), who showed that many earlier molecular graph networks fit this pattern and trained them on DFT properties of the QM9 molecules. Models in this family (SchNet, DimeNet, equivariant networks such as NequIP or MACE that also use 3D geometry) are now widely used to predict molecular properties and fit interatomic potentials, but they are only as good as their training data and can fail on chemistry unlike it. Try 1 layer (only nearest neighbours), fewer training molecules, or a narrower network.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/molecular_gnn/main.py
"""Main loop: random molecules → Hückel targets → train GNN and MLP → test on larger molecules → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

N_PAD = 16  # every molecule is padded to 16 atoms (the largest test size)
TRAIN_SIZES = (4, 10)  # atoms per training molecule
LARGE_SIZES = (11, 16)  # extrapolation test: larger molecules than any seen in training
BATCH = 128
SEED = 0


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    n_train = 400 if fast else params["n_train"]
    n_test = 100 if fast else 400
    epochs = 10 if fast else params["epochs"]
    t0 = time.perf_counter()

    # --- data: graphs and their exact Hückel π energies ---
    train = tensors.dataset(n_train, TRAIN_SIZES, N_PAD, SEED, dev)
    small = tensors.dataset(n_test, TRAIN_SIZES, N_PAD, SEED + 1, dev)  # same sizes, unseen molecules
    large = tensors.dataset(n_test, LARGE_SIZES, N_PAD, SEED + 2, dev)  # bigger molecules
    y = {
        name: equation.huckel_energy(d["A"], d["mask"])
        for name, d in [("train", train), ("small", small), ("large", large)]
    }
    tests = {"small": small, "large": large}

    # --- train the GNN and the MLP on the same molecules ---
    w_gnn = tensors.init_gnn(params["width"], params["layers"], SEED, dev)
    w_mlp = tensors.init_mlp(N_PAD, 128, SEED, dev)
    t1 = time.perf_counter()
    h_gnn = solve.train(
        equation.gnn, w_gnn, {**train, "y": y["train"]}, tests, epochs, params["lr"], BATCH, SEED, "GNN epochs"
    )
    t_gnn = time.perf_counter() - t1
    t1 = time.perf_counter()
    h_mlp = solve.train(equation.mlp, w_mlp, {**train, "y": y["train"]}, tests, epochs, 1e-3, BATCH, SEED, "MLP epochs")
    t_mlp = time.perf_counter() - t1

    # --- baselines and the permutation test ---
    _, fit = equation.count_baseline(train, y["train"])
    perm = torch.randperm(N_PAD, generator=torch.Generator().manual_seed(SEED + 3)).to(dev)
    with torch.no_grad():
        y_gnn, e_gnn = equation.gnn(large, w_gnn, return_atoms=True)
        gnn_perm = (equation.gnn(tensors.permute(large, perm), w_gnn) - y_gnn).abs().max().item()
        mlp_perm = (equation.mlp(tensors.permute(large, perm), w_mlp) - equation.mlp(large, w_mlp)).abs().max().item()

    def mae(pred, name):
        return (pred - y[name]).abs().mean().item()

    print(f"{n_train} training molecules with {TRAIN_SIZES[0]}–{TRAIN_SIZES[1]} carbons, test sets of {n_test}: "
          f"same sizes and {LARGE_SIZES[0]}–{LARGE_SIZES[1]} carbons   ·   y = π energy in units of |β|")  # fmt: skip
    print(f"GNN: {params['layers']} message-passing layers × {params['width']} channels, {tensors.count(w_gnn):,} "
          f"weights, {t_gnn:.1f} s   ·   MLP on the 16×16 adjacency: {tensors.count(w_mlp):,} weights, "
          f"{t_mlp:.1f} s")  # fmt: skip
    print("mean absolute error (|β|)        same-size test   larger molecules")
    rows = [
        ("predict the training mean", torch.full_like(y["small"], y["train"].mean().item()),
         torch.full_like(y["large"], y["train"].mean().item())),
        ("bond counting  a·N + b·N_bonds + c", equation.count_baseline(small, y["small"], fit)[0],
         equation.count_baseline(large, y["large"], fit)[0]),
        ("MLP on the padded adjacency", h_mlp["small"][-1].to(dev), h_mlp["large"][-1].to(dev)),
        ("message-passing GNN", h_gnn["small"][-1].to(dev), h_gnn["large"][-1].to(dev)),
    ]  # fmt: skip
    for name, p_small, p_large in rows:
        print(f"  {name:<34} {mae(p_small, 'small'):10.3f} {mae(p_large, 'large'):16.3f}")
    print(f"training MSE: GNN {h_gnn['loss'][0].item():.2f} → {h_gnn['loss'][-1].item():.4f},  MLP "
          f"{h_mlp['loss'][0].item():.2f} → {h_mlp['loss'][-1].item():.4f}   "
          f"(variance of y: {y['train'].var().item():.2f})")  # fmt: skip
    print(f"relabel the atoms of every large molecule: GNN output changes by ≤ {gnn_perm:.1e}, MLP by up to "
          f"{mlp_perm:.2f}   ·   total {time.perf_counter() - t0:.1f} s")  # fmt: skip

    m = int(torch.argmax(large["mask"].sum(-1) * 10 + large["A"].sum((-1, -2))))  # biggest, most-ringed molecule
    example = {"A": large["A"][m], "mask": large["mask"][m], "e": e_gnn[m], "y": y["large"][m].item(),
               "pred": y_gnn[m].item(), "gnn_perm": gnn_perm, "mlp_perm": mlp_perm}  # fmt: skip
    return plot.animate(h_gnn, h_mlp, y, y["train"].var().item(), example, out_dir / "molecular_gnn.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/molecular_gnn/equation.py
r"""Physics → tensor translation.

THE TARGET (a known rule, so the data are exact): the Hückel π-electron energy. Each sp² carbon contributes one
p orbital and one π electron; the Hückel Hamiltonian is H = α 1 + β A (A = adjacency of the carbon skeleton,
β < 0). With eigenvalues λ_1 ≥ λ_2 ≥ … of A and the N electrons filling the lowest-energy (highest-λ) orbitals
two at a time:

    E_π = N α + β y,      y = Σ_k o_k λ_k      (o_k = 2, 2, …, then 1 for an odd electron, then 0)

We learn y, the π binding energy in units of |β|, from the graph alone.

THE MODEL: a message-passing neural network (MPNN). Indices: m = molecule, i/j = atom, c/d = channel.

    embed      h⁰_{mic}    = σ(x_{mif} W_{fc} + b_c) · mask_{mi}                      f summed
    message    s_{mic}     = A_{mij} h_{mjc}                                           j summed: neighbours only
    update     h'_{mid}    = σ(h_{mic} U_{cd} + s_{mic} V_{cd} + c_d) · mask_{mi}       c summed
    readout    y_m         = Σ_i mask_{mi} e_{mi},    e_{mi} = Q2_d σ(h_{mic} Q1_{cd} + q1_d) + q2

Every weight acts on ONE atom at a time and the only sum over atoms is through A_{mij} or the final Σ_i, so
relabelling the atoms (A → ΠAΠᵀ, x → Πx) leaves y unchanged, and the same weights work for any molecule size.
"""

import torch

# --- the target rule ---


def huckel_energy(A: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    r"""y_m = Σ_k o_k λ_{mk}: π binding energy in units of |β| for every padded molecule.   (M, N, N) → (M,).

    Padded atoms get a huge negative diagonal so their eigenvalues sink to the bottom and stay unoccupied.
    """
    A = A.double()
    pad = torch.diag_embed(-100.0 * (1 - mask.double()))
    lam = torch.linalg.eigvalsh(A + pad).flip(-1)  # (M, N) eigenvalues, largest first
    n_electrons = mask.sum(-1, keepdim=True).double()  # one π electron per carbon
    k = torch.arange(A.shape[-1], device=A.device, dtype=torch.float64)
    occupation = (n_electrons - 2 * k).clamp(0, 2)  # 2, 2, …, (1), 0, 0
    return torch.einsum("mk,mk->m", occupation, lam).to(mask.dtype)


# --- the graph neural network ---


def embed(x: torch.Tensor, mask: torch.Tensor, W: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """h⁰_{mic} = σ(x_{mif} W_{fc} + b_c) · mask_{mi}   (f summed).   (M, N, F) → (M, N, C)."""
    return torch.relu(torch.einsum("mif,fc->mic", x, W) + b) * mask[..., None]


def message(A: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
    """s_{mic} = A_{mij} h_{mjc}   (j summed): each atom adds up its neighbours' states.   (M, N, C)."""
    return torch.einsum("mij,mjc->mic", A, h)


def update(h, s, mask, U, V, c) -> torch.Tensor:
    """h'_{mid} = σ(h_{mic} U_{cd} + s_{mic} V_{cd} + c_d) · mask_{mi}   (c summed), plus a residual h."""
    out = torch.einsum("mic,cd->mid", h, U) + torch.einsum("mic,cd->mid", s, V) + c
    return (h + torch.relu(out)) * mask[..., None]


def atom_contributions(h: torch.Tensor, Q1, q1, Q2, q2) -> torch.Tensor:
    """e_{mi} = Q2_d σ(h_{mic} Q1_{cd} + q1_d) + q2: a per-atom energy.   (M, N, C) → (M, N)."""
    return torch.einsum("mid,d->mi", torch.relu(torch.einsum("mic,cd->mid", h, Q1) + q1), Q2) + q2


def gnn(data: dict, w: dict, return_atoms: bool = False):
    """y_m = Σ_i mask_{mi} e_{mi} after len(U) rounds of message passing.   → (M,) (and e (M, N))."""
    A, mask = data["A"], data["mask"]
    h = embed(data["x"], mask, w["W_in"], w["b_in"])
    for U, V, c in zip(w["U"], w["V"], w["c"]):
        h = update(h, message(A, h), mask, U, V, c)
    e = atom_contributions(h, w["Q1"], w["q1"], w["Q2"], w["q2"]) * mask
    y = e.sum(-1)  # Σ_i: a size-extensive, permutation-invariant readout
    return (y, e) if return_atoms else y


# --- the baseline ---


def mlp(data: dict, w: dict) -> torch.Tensor:
    """Fixed-size MLP on the flattened padded adjacency a_m = vec(A_m) ∈ R^{N²}: no notion of atoms or relabelling."""
    a = data["A"].flatten(1)  # (M, N²)
    h = torch.relu(a @ w["W1"] + w["b1"])
    h = torch.relu(h @ w["W2"] + w["b2"])
    return h @ w["W3"] + w["b3"]


def mse(pred: torch.Tensor, true: torch.Tensor) -> torch.Tensor:
    return ((pred - true) ** 2).mean()


def count_baseline(data: dict, y: torch.Tensor, fit: torch.Tensor | None = None):
    """Linear 'bond counting' model y ≈ a N_atoms + b N_bonds + c, fitted by least squares on the training set."""
    n_atoms = data["mask"].sum(-1)
    n_bonds = data["A"].sum((-1, -2)) / 2
    X = torch.stack([n_atoms, n_bonds, torch.ones_like(n_atoms)], dim=-1)  # (M, 3)
    if fit is None:
        fit = torch.linalg.lstsq(X.double(), y[:, None].double()).solution[:, 0].to(y.dtype)
    return X @ fit, fit


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/molecular_gnn/tensors.py
"""Build the concrete tensors: random conjugated hydrocarbon graphs (padded to N atoms), atom features, weights.

A molecule is its hydrogen-suppressed carbon skeleton: atoms are sp² carbons (one p orbital each), bonds are
C–C σ bonds. Skeletons are grown at random as trees with at most 3 neighbours per carbon, then closed into
5- and 6-membered rings. They are graph-theoretic toys, not a curated set of real molecules.

Padding: a batch of molecules with up to N atoms is stored as
    A (M, N, N)  adjacency, 0 for padded atoms         mask (M, N)  1 for real atoms, 0 for padding
    x (M, N, F)  atom features (one-hot number of carbon neighbours 1, 2, 3), 0 for padding
"""

import math
import random

import torch

MAX_DEGREE = 3  # an sp² carbon has three σ bonds (to C or H)
N_FEATURES = MAX_DEGREE


def _distance(adj: list[set[int]], start: int) -> list[int]:
    """Breadth-first graph distances from one atom."""
    dist = [-1] * len(adj)
    dist[start], frontier = 0, [start]
    while frontier:
        nxt = []
        for i in frontier:
            for j in adj[i]:
                if dist[j] < 0:
                    dist[j] = dist[i] + 1
                    nxt.append(j)
        frontier = nxt
    return dist


def random_skeleton(n: int, rng: random.Random) -> list[set[int]]:
    """A random connected carbon skeleton with n atoms: a tree with degree ≤ 3, plus ring closures that make
    5- or 6-membered rings (bonds between atoms at graph distance 4 or 5)."""
    adj = [set() for _ in range(n)]
    for new in range(1, n):  # attach each new atom to an earlier atom that still has a free valence
        free = [i for i in range(new) if len(adj[i]) < MAX_DEGREE]
        parent = rng.choice(free)
        adj[new].add(parent)
        adj[parent].add(new)
    for _ in range(rng.randint(0, n // 4)):  # a few ring closures
        free = [i for i in range(n) if len(adj[i]) < MAX_DEGREE]
        pairs = [(i, j) for i in free for j in free if i < j and _distance(adj, i)[j] in (4, 5)]
        if not pairs:
            break
        i, j = rng.choice(pairs)
        adj[i].add(j)
        adj[j].add(i)
    return adj


def dataset(n_mol: int, sizes: tuple[int, int], n_pad: int, seed: int, device: torch.device) -> dict:
    """n_mol random molecules with sizes[0] … sizes[1] atoms, padded to n_pad: A (M, N, N), mask (M, N), x (M, N, F)."""
    rng = random.Random(seed)
    A = torch.zeros(n_mol, n_pad, n_pad, dtype=torch.float32)
    for m in range(n_mol):
        n = rng.randint(*sizes)
        for i, nbrs in enumerate(random_skeleton(n, rng)):
            A[m, i, list(nbrs)] = 1.0
    mask = (A.sum(-1) > 0).to(torch.float32)  # every real atom has at least one bond (n ≥ 2)
    return {"A": A.to(device), "mask": mask.to(device), "x": features(A, mask).to(device)}


def features(A: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """x_{mif} = 1 if atom i has f+1 carbon neighbours (f = 0, 1, 2): (M, N, N) → (M, N, F)."""
    degree = A.sum(-1).long()  # (M, N)
    x = torch.nn.functional.one_hot((degree - 1).clamp(0, N_FEATURES - 1), N_FEATURES).to(A.dtype)
    return x * mask[..., None]


def permute(data: dict, perm: torch.Tensor) -> dict:
    """Relabel the atoms of every molecule: A' = Π A Πᵀ, x' = Π x, mask' = Π mask (same permutation for all)."""
    return {"A": data["A"][:, perm][:, :, perm], "mask": data["mask"][:, perm], "x": data["x"][:, perm]}


def init_gnn(width: int, n_layers: int, seed: int, device: torch.device) -> dict:
    """GNN weights: embed W_in (F, C); per layer U[l], V[l] (C, C), c[l] (C,); readout Q1 (C, C), Q2 (C,)."""
    gen = torch.Generator().manual_seed(seed)

    def dense(fan_in, *shape):
        return (torch.rand(*shape, generator=gen, dtype=torch.float32) * 2 - 1) / math.sqrt(fan_in)

    w = {
        "W_in": dense(N_FEATURES, N_FEATURES, width),
        "b_in": torch.zeros(width, dtype=torch.float32),
        "U": [dense(width, width, width) for _ in range(n_layers)],
        "V": [dense(width, width, width) * 0.5 for _ in range(n_layers)],
        "c": [torch.zeros(width, dtype=torch.float32) for _ in range(n_layers)],
        "Q1": dense(width, width, width),
        "q1": torch.zeros(width, dtype=torch.float32),
        "Q2": dense(width, width),
        "q2": torch.zeros((), dtype=torch.float32),
    }
    return _trainable(w, device)


def init_mlp(n_pad: int, hidden: int, seed: int, device: torch.device) -> dict:
    """Baseline MLP on the flattened padded adjacency: (N², H), (H, H), (H,) weights."""
    gen = torch.Generator().manual_seed(seed + 1)

    def dense(fan_in, *shape):
        return (torch.rand(*shape, generator=gen, dtype=torch.float32) * 2 - 1) / math.sqrt(fan_in)

    w = {
        "W1": dense(n_pad * n_pad, n_pad * n_pad, hidden),
        "b1": torch.zeros(hidden, dtype=torch.float32),
        "W2": dense(hidden, hidden, hidden),
        "b2": torch.zeros(hidden, dtype=torch.float32),
        "W3": dense(hidden, hidden),
        "b3": torch.zeros((), dtype=torch.float32),
    }
    return _trainable(w, device)


def _trainable(w: dict, device: torch.device) -> dict:
    return {
        k: [t.to(device).requires_grad_(True) for t in v] if isinstance(v, list) else v.to(device).requires_grad_(True)
        for k, v in w.items()
    }


def parameters(w: dict) -> list[torch.Tensor]:
    return [t for v in w.values() for t in (v if isinstance(v, list) else [v])]


def count(w: dict) -> int:
    return sum(t.numel() for t in parameters(w))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/molecular_gnn/solve.py
"""Training loop (Adam on the mean squared error), shared by the GNN and the MLP baseline."""

import torch

from problems._common.progress import steps

from .equation import mse
from .tensors import parameters


def subset(data: dict, idx: torch.Tensor) -> dict:
    return {k: v[idx] for k, v in data.items()}


def train(model, w: dict, train_set: dict, tests: dict, epochs: int, lr: float, batch: int, seed: int, label: str):
    """Fit model(data, w) to train_set["y"]. Returns, per epoch (entry 0 = untrained), the training MSE and the
    predictions on every test set in `tests` (name → data dict with "y")."""
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(parameters(w), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=lr / 30)
    inputs = {k: v for k, v in train_set.items() if k != "y"}
    y = train_set["y"]

    def record(loss: float) -> dict:
        with torch.no_grad():
            return {"loss": loss, **{name: model(d, w).cpu() for name, d in tests.items()}}

    with torch.no_grad():
        history = [record(mse(model(inputs, w), y).item())]
    for _ in steps(epochs, label):
        order = torch.randperm(len(y), generator=gen).to(y.device)
        losses = []
        for start in range(0, len(y), batch):
            idx = order[start : start + batch]
            loss = mse(model(subset(inputs, idx), w), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sched.step()
        history.append(record(sum(losses) / len(losses)))
    out = {"loss": torch.tensor([h["loss"] for h in history])}
    for name in tests:
        out[name] = torch.stack([h[name] for h in history])  # (epochs + 1, M_test)
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/molecular_gnn/plot.py
"""Animate training: loss curves, predicted vs true π energy for the GNN and the MLP, and one large test molecule."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

BLUE, RED, GREY, DARK = "#2563eb", "#dc2626", "#9ca3af", "#374151"


def animate(h_gnn: dict, h_mlp: dict, y: dict, var: float, ex: dict, path: Path) -> Path:
    y_s, y_l = y["small"].cpu().numpy(), y["large"].cpu().numpy()
    g_s, g_l = h_gnn["small"].numpy(), h_gnn["large"].numpy()  # (E+1, M)
    m_s, m_l = h_mlp["small"].numpy(), h_mlp["large"].numpy()
    loss_g, loss_m = h_gnn["loss"].numpy(), h_mlp["loss"].numpy()
    big_g = ((g_l - y_l) ** 2).mean(1)  # test MSE on the larger molecules, per epoch
    big_m = ((m_l - y_l) ** 2).mean(1)
    epochs = np.arange(len(loss_g))
    lo, hi = 0.0, max(y_l.max(), y_s.max()) * 1.08

    fig = plt.figure(figsize=(8.4, 4.8))
    grid = fig.add_gridspec(2, 3, width_ratios=[1.1, 1, 1])
    ax_e = fig.add_subplot(grid[0, 0])
    ax_m = fig.add_subplot(grid[1, 0])
    ax_g = fig.add_subplot(grid[:, 1])
    ax_p = fig.add_subplot(grid[:, 2])
    for ax in (ax_g, ax_p):
        ax.set_aspect("equal")

    # --- loss curves ---
    ax_e.set_yscale("log")
    ax_e.minorticks_off()
    ax_e.set_xlim(0, epochs[-1])
    ax_e.set_ylim(min(loss_g.min(), loss_m.min()) * 0.5, max(big_m[-1], var) * 60)  # legend room; clips epoch 0
    ax_e.axhline(var, color=GREY, ls="--", lw=1)
    ax_e.text(epochs[-1], var * 1.15, "predict the mean", ha="right", va="bottom", fontsize=6.5, color="#6b7280")
    lines = {
        "gt": ax_e.plot([], [], color=BLUE, lw=1.5, label="GNN train")[0],
        "gl": ax_e.plot([], [], color=BLUE, lw=1.2, ls=":", label="GNN larger mol.")[0],
        "mt": ax_e.plot([], [], color=RED, lw=1.5, label="MLP train")[0],
        "ml": ax_e.plot([], [], color=RED, lw=1.2, ls=":", label="MLP larger mol.")[0],
    }
    ax_e.set_title("mean squared error", fontsize=9)
    ax_e.tick_params(labelsize=7)
    ax_e.legend(fontsize=6, loc="upper right", ncol=2)

    # --- predicted vs true ---
    scat = {}
    for ax, name in ((ax_g, "message-passing GNN"), (ax_p, "MLP on adjacency")):
        ax.plot([lo, hi], [lo, hi], color=GREY, lw=1)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_xlabel("true π energy  y / |β|", fontsize=8)
        ax.set_title(name, fontsize=9)
        ax.tick_params(labelsize=7)
        scat[ax] = (
            ax.plot([], [], "o", color=BLUE, ms=2.2, alpha=0.6, label="4–10 C (as trained)")[0],
            ax.plot([], [], "o", color=RED, ms=2.2, alpha=0.6, label="11–16 C (larger)")[0],
        )
    ax_g.set_ylabel("predicted", fontsize=8)
    ax_g.legend(fontsize=6.5, loc="upper left", markerscale=2)
    mae_txt = {ax: ax.text(0.97, 0.03, "", transform=ax.transAxes, ha="right", fontsize=7) for ax in (ax_g, ax_p)}

    # --- one large test molecule, atoms coloured by their learned contribution e_i ---
    _draw_molecule(ax_m, ex)
    counter = fig.text(0.5, 0.005, "", ha="center", va="bottom", fontsize=9)

    def update(k: int):
        lines["gt"].set_data(epochs[: k + 1], loss_g[: k + 1])
        lines["gl"].set_data(epochs[: k + 1], big_g[: k + 1])
        lines["mt"].set_data(epochs[: k + 1], loss_m[: k + 1])
        lines["ml"].set_data(epochs[: k + 1], big_m[: k + 1])
        for ax, ps, pl in ((ax_g, g_s, g_l), (ax_p, m_s, m_l)):
            scat[ax][0].set_data(y_s, ps[k])
            scat[ax][1].set_data(y_l, pl[k])
            mae_txt[ax].set_text(f"MAE {np.abs(ps[k] - y_s).mean():.2f} (blue), {np.abs(pl[k] - y_l).mean():.2f} (red)")
        counter.set_text(f"epoch {k}   ·   GNN training MSE {loss_g[k]:.3f}   ·   MLP {loss_m[k]:.3f}")

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, frame_indices(len(epochs), 60), path, fps=8)


def _draw_molecule(ax, ex: dict) -> None:
    mask = ex["mask"].cpu().numpy() > 0
    A = ex["A"].cpu().numpy()[np.ix_(mask, mask)]
    e = ex["e"].cpu().numpy()[mask]
    xy = _layout(A)
    for i, j in zip(*np.nonzero(np.triu(A))):
        ax.plot(xy[[i, j], 0], xy[[i, j], 1], color=DARK, lw=1.5, zorder=1)
    sc = ax.scatter(xy[:, 0], xy[:, 1], c=e, cmap="Blues", s=60, edgecolors="#1e40af", linewidths=0.8, zorder=2)
    cb = plt.colorbar(sc, ax=ax, fraction=0.05, pad=0.02)
    cb.ax.tick_params(labelsize=6)
    cb.set_label("$e_i$", fontsize=7)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"{mask.sum()} C: true {ex['y']:.2f}, GNN {ex['pred']:.2f}", fontsize=8)
    ax.text(0.5, -0.08, f"relabel atoms: GNN Δ ≤ {ex['gnn_perm']:.0e}, MLP Δ = {ex['mlp_perm']:.1f}",
            transform=ax.transAxes, ha="center", va="top", fontsize=6.5)  # fmt: skip


def _layout(A: np.ndarray) -> np.ndarray:
    """2D coordinates from graph distances by classical multidimensional scaling (bonded atoms end up close)."""
    n = len(A)
    D = np.where(A > 0, 1.0, np.inf)
    np.fill_diagonal(D, 0.0)
    for k in range(n):  # Floyd–Warshall shortest paths
        D = np.minimum(D, D[:, [k]] + D[[k], :])
    J = np.eye(n) - 1.0 / n
    vals, vecs = np.linalg.eigh(-0.5 * J @ (D**2) @ J)
    return vecs[:, -2:] * np.sqrt(np.maximum(vals[-2:], 1e-9))


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_train': 2000,
    'epochs': 120,
    'width': 32,
    'layers': 4,
    'lr': 0.005
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.molecular_gnn")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.molecular_gnn.main")
problem_dir = Path("problems/chemistry/molecular_gnn")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
